In [19]:
import pandas as pd
import numpy as np

In [18]:
url = "https://raw.githubusercontent.com/EveliaCoss/CAMDA2025_metadatos/refs/heads/main/rawdata/TrainAndTest_cleaned/training_metadata_cleaned.tsv"
df = pd.read_csv(url, sep='\t')

In [2]:
df.to_csv("training_metadata_cleaned.csv", index=False)

In [39]:
def process_MIC_data(input_file, output_file):
    """
    Procesa un archivo CSV con datos MIC (Concentración Mínima Inhibitoria),
    recategoriza los valores MIC y asigna un fenotipo (Susceptible o Resistente)
    basado en tablas interpretativas específicas para cada género. Si no se puede
    recategorizar el MIC, se usa el fenotipo ya proporcionado en la columna 'phenotype'.

    Parámetros:
    -----------
    input_file : str
        Ruta al archivo CSV de entrada que contiene las columnas 'measurement_value', 
        'accession', 'new_genus' y 'phenotype' (opcional pero recomendada).

    output_file : str
        Ruta del archivo CSV donde se guardará el DataFrame procesado.

    Retorna:
    --------
    df : pandas.DataFrame
        El DataFrame resultante con las columnas nuevas 'recategorized_MIC' y 'phenotype_assigned'.
    """

    import pandas as pd
    import numpy as np

    df = pd.read_csv(input_file)
    
    # Asegurar que measurement_value sea numérico
    df["measurement_value"] = pd.to_numeric(df["measurement_value"], errors="coerce")

    # Eliminar accesiones no deseadas
    accessions_to_remove = [
        "ERR1218638", "ERR1218722", "SRR2101499", "SRR960879",
        "SRR850995", "ERR1218771", "SRR5386043", "SRR6985679"
    ]
    df = df[~df["accession"].isin(accessions_to_remove)]

    # Definir rangos y etiquetas para categorizar MIC
    bins = [0, 0.09, 0.185, 0.375, 0.75, 1.5, 3, 6, 12, 24, 48, 10000]
    labels = [0.06, 0.12, 0.25, 0.5, 1, 2, 4, 8, 16, 32, 64]
    recategorized_MIC = pd.cut(df["measurement_value"], bins=bins, labels=labels, right=False)

    df.insert(df.columns.get_loc("measurement_value") + 1, "recategorized_MIC", recategorized_MIC)

    # Tabla de interpretación por género
    phenotype_table = {
        "Klebsiella":              list('sssssssrrrr'),
        "Escherichia":             list('sssssssrrrr'),
        "Salmonella":              list('sssssssrrrr'),
        "Streptococcus":           list('ssssrrrrrrr'),
        "Staphylococcus":          list('sssssssrrrr'),
        "Pseudomonas":             list('sssssssssrr'),
        "Acinetobacter":           list('ssssssssrrr'),
        "Campylobacter":           list('ssssssssrrr'),
        "Neisseria":               list('sssssrrrrrr')
    }

    phenotype_df = pd.DataFrame(phenotype_table, index=labels).T

    # Función auxiliar para asignar fenotipo
    def assign_phenotype(row):
        genus = row["new_genus"]
        mic = row["recategorized_MIC"]
        if not pd.isna(mic) and genus in phenotype_df.index:
            return "Susceptible" if phenotype_df.loc[genus, mic] == "s" else "Resistant"
        elif pd.isna(mic) and not pd.isna(row.get("phenotype")):
            return row["phenotype"].capitalize()  # Usa el valor original si está disponible
        else:
            return np.nan

    # Asignar fenotipo
    df["phenotype_assigned"] = df.apply(assign_phenotype, axis=1)

    # Reordenar para insertar justo después de new_genus
    df.insert(df.columns.get_loc("new_genus") + 1, "phenotype_assigned", df.pop("phenotype_assigned"))

    df.to_csv(output_file, index=False)
    return df


In [40]:
df_resultado = process_MIC_data(
    input_file="training_metadata_cleaned.csv",
    output_file="CAMDA25_training_con_MIC_y_fenotipo_1.csv"
)
